In [1]:
import os
from glob import glob
import torch
import numpy as np
X_train = torch.from_numpy(np.stack([np.loadtxt(file, dtype=np.float32) for file in glob(os.path.join("train/Inertial Signals", "*.txt"))], axis=-1))
y_train = torch.from_numpy(np.loadtxt("train/y_train.txt", dtype=np.int64)) - 1
X_test = torch.from_numpy(np.stack([np.loadtxt(file, dtype=np.float32) for file in glob(os.path.join("test/Inertial Signals", "*.txt"))], axis=-1))
y_test = torch.from_numpy(np.loadtxt("test/y_test.txt", dtype=np.int64)) - 1 
X_train.shape, y_train.shape, X_test.shape, y_test.shape

(torch.Size([7352, 128, 9]),
 torch.Size([7352]),
 torch.Size([2947, 128, 9]),
 torch.Size([2947]))

In [ ]:
from torch.utils.data import TensorDataset, DataLoader, random_split
batch_size = 128

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)
train_size = int(0.8*len(train_dataset))
valid_size = len(train_dataset) - train_size
train_dataset, valid_dataset = random_split(train_dataset, [train_size, valid_size])
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

from collections import Counter
counter = Counter()
for x, y in train_loader:
    counter.update(y.tolist())
print(counter)

In [18]:
from torchinfo import summary
from ResCBAR import ResCBAR
from stateSpaceModel import StateSpaceModel
model = StateSpaceModel(6, 128, 64, 4, dropout=0.1).to('cuda')
#model = ResCBAR(6, 128, 128).to('cuda')
summary(model, input_size=(64, 128, 9))

import torch.nn as nn
from tqdm.auto import tqdm

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
epochs = 20

for epoch in range(epochs):

    train_losses, valid_losses = [], []
    train_correct, train_total = 0, 0
    valid_correct, valid_total = 0, 0

    # --------------------
    # Train
    # --------------------
    model.train()
    for train, labels in tqdm(train_loader):
        optimizer.zero_grad()
        outputs = model(train.to('cuda')).to('cpu')
        train_loss = criterion(outputs, labels)
        train_loss.backward()
        optimizer.step()
        train_losses.append(train_loss.item())

        # Accuracy
        _, predicted = torch.max(outputs, dim=1)
        train_correct += (predicted == labels).sum().item()
        train_total += labels.size(0)

    train_acc = train_correct / train_total
    print(f"Train: Epoch [{epoch+1}/{epochs}], Loss: {torch.tensor(train_losses).mean():.4f}, Accuracy: {train_acc:.4f}")

    # --------------------
    # Test
    # --------------------
    model.eval()
    with torch.no_grad():
        for test, labels in tqdm(valid_loader):
            outputs = model(test.to('cuda')).to('cpu')
            valid_loss = criterion(outputs, labels)
            valid_losses.append(valid_loss.item())

            # Accuracy
            predicted = torch.argmax(outputs, dim=1)
            valid_correct += (predicted == labels).sum().item()
            valid_total += labels.size(0)

    valid_acc = valid_correct / valid_total
    print(f"Valid: Epoch [{epoch+1}/{epochs}], Loss: {torch.tensor(valid_losses).mean():.4f}, Accuracy: {valid_acc:.4f}")


  0%|          | 0/46 [00:00<?, ?it/s]

Train: Epoch [1/20], Loss: 1.5089, Accuracy: 0.2795


  0%|          | 0/12 [00:00<?, ?it/s]

Valid: Epoch [1/20], Loss: 1.0932, Accuracy: 0.3651


  0%|          | 0/46 [00:00<?, ?it/s]

...

Valid: Epoch [19/20], Loss: 0.1430, Accuracy: 0.9470


  0%|          | 0/46 [00:00<?, ?it/s]

Train: Epoch [20/20], Loss: 0.0959, Accuracy: 0.9638


  0%|          | 0/12 [00:00<?, ?it/s]

Valid: Epoch [20/20], Loss: 0.1485, Accuracy: 0.9524


In [19]:
test_losses, test_correct, test_total = [], 0, 0
model.eval()
with torch.no_grad():
    for test, labels in tqdm(test_loader):
        outputs = model(test.to('cuda')).to('cpu')
        test_loss = criterion(outputs, labels)
        test_losses.append(test_loss.item())

        # Accuracy
        _, predicted = torch.max(outputs, dim=1)
        test_correct += (predicted == labels).sum().item()
        test_total += labels.size(0)

test_acc = test_correct / test_total
print(f"Test: Epoch [{epoch+1}/{epochs}], Loss: {torch.tensor(test_losses).mean():.4f}, Accuracy: {test_acc:.4f}")

  0%|          | 0/24 [00:00<?, ?it/s]

Test: Epoch [20/20], Loss: 0.2446, Accuracy: 0.9199
